# Init

In [0]:
import pyspark.sql.functions as F 
from pyspark.sql.window import Window

#The Transformation Logic

In [0]:
query = """
-- Deduplicate dimension tables first to prevent cartesian products
WITH dedupe_products AS (
  SELECT 
    product_number,
    product_key,
    ROW_NUMBER() OVER (PARTITION BY product_number ORDER BY product_key DESC) as rn
  FROM workspace_1.gold.dim_products
),
dedupe_customers AS (
  SELECT 
    customer_id,
    customer_key,
    ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY customer_key DESC) as rn
  FROM workspace_1.gold.dim_customers
)
SELECT
    sd.order_number,
    pr.product_key,
    cu.customer_key,
    sd.order_date,
    sd.ship_date,
    sd.due_date,
    sd.sales_amount,
    sd.quantity,
    sd.price
FROM workspace_1.silver.crm_sales sd
LEFT JOIN dedupe_products pr
    ON sd.product_number = pr.product_number AND pr.rn = 1
LEFT JOIN dedupe_customers cu
    ON sd.customer_id = cu.customer_id AND cu.rn = 1
WHERE sd.sales_amount IS NULL OR sd.sales_amount >= 0;  -- Filter out only negative sales, keep nulls
"""
df = spark.sql(query)

In [0]:
df.limit(10).display()

# Writing Gold Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace_1.gold.fact_sales")

# Data Quality Framework

## Multi-Layer Quality Validation for:
* **Data Scientists** - ML readiness, nulls, outliers, cardinality
* **Data Analysts** - Business logic, reconciliation, referential integrity
* **Business Users** - Freshness, completeness, trust metrics

---

### Quality Checks Run in Order:
1. **Bronze Layer** - Source data validation
2. **Silver Layer** - Transformation validation  
3. **Gold Layer** - Business metrics validation

### Alert Strategy:
* 🔴 **CRITICAL** - Pipeline fails immediately
* 🟡 **WARNING** - Pipeline continues with alert
* 🟢 **INFO** - Logged for monitoring

In [0]:
# ============================================
# CONFIGURATION & MONITORING SETUP
# ============================================

from pyspark.sql.functions import col, current_timestamp
from datetime import datetime
import json

# Quality thresholds
QUALITY_CONFIG = {
    "pipeline_name": "Bike_Lakehouse",
    "task_name": "Gold_Fact_Sales",
    "layer": "GOLD",
    
    "thresholds": {
        "revenue_match_tolerance_pct": 0.01,  # 0.01% tolerance
        "max_null_foreign_keys_pct": 0.0,     # 0% allowed
    },
    
    "monitoring_table": "workspace_1.monitoring.data_quality_log",
}

# Create monitoring schema and table
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace_1.monitoring")

monitoring_ddl = """
CREATE TABLE IF NOT EXISTS workspace_1.monitoring.data_quality_log (
    run_timestamp TIMESTAMP,
    pipeline_name STRING,
    task_name STRING,
    layer STRING,
    quality_score INT,
    status STRING,
    total_records BIGINT,
    data_freshness_hours INT,
    null_product_keys INT,
    null_customer_keys INT,
    outlier_count INT,
    unique_products INT,
    unique_customers INT,
    unique_orders INT,
    revenue_reconciliation_pass BOOLEAN,
    revenue_gold DECIMAL(18,2),
    revenue_silver DECIMAL(18,2),
    revenue_diff_pct DECIMAL(10,4),
    record_count_pass BOOLEAN,
    negative_sales INT,
    invalid_ship_dates INT,
    invalid_due_dates INT,
    price_calculation_errors INT,
    orphaned_products INT,
    orphaned_customers INT,
    last_order_date DATE,
    data_age_days INT,
    completeness_pct DECIMAL(5,2),
    critical_issues ARRAY<STRING>,
    warnings ARRAY<STRING>,
    full_metrics STRING
)
USING DELTA
COMMENT 'Data quality metrics log for all pipeline tasks'
"""

spark.sql(monitoring_ddl)

print("✅ Configuration loaded")
print(f"   Pipeline: {QUALITY_CONFIG['pipeline_name']}")
print(f"   Task: {QUALITY_CONFIG['task_name']}")
print(f"   Monitoring: {QUALITY_CONFIG['monitoring_table']}")

In [0]:
%sql
-- ============================================
-- REVENUE RECONCILIATION CHECK
-- ============================================
-- WHAT: Compares total revenue (SUM of sales_amount) between Gold and Silver layers
-- WHY: Ensures no revenue was LOST during transformation (bad joins dropping records)
--      or DUPLICATED (cartesian products inflating totals)
-- CRITICAL: Revenue mismatch means financial reports will be WRONG
-- THRESHOLD: Must match within 0.01% (tolerates rounding errors only)

WITH gold_revenue AS (
  SELECT SUM(sales_amount) as total_revenue
  FROM workspace_1.gold.fact_sales
),
silver_revenue AS (
  SELECT SUM(sales_amount) as total_revenue
  FROM workspace_1.silver.crm_sales
)
SELECT 
  'Revenue Reconciliation' as check_name,
  g.total_revenue as gold_revenue,
  s.total_revenue as silver_revenue,
  g.total_revenue - s.total_revenue as difference,
  ABS(g.total_revenue - s.total_revenue) / s.total_revenue * 100 as diff_pct,
  CASE 
    WHEN ABS(g.total_revenue - s.total_revenue) / s.total_revenue * 100 <= 0.01 
    THEN '✅ PASS'
    ELSE '❌ FAIL'
  END as status,
  CASE 
    WHEN ABS(g.total_revenue - s.total_revenue) / s.total_revenue * 100 <= 0.01 
    THEN 'Revenue matches within tolerance'
    ELSE CONCAT('Revenue mismatch: ', CAST(ABS(g.total_revenue - s.total_revenue) / s.total_revenue * 100 AS DECIMAL(10,4)), '% difference exceeds 0.01% threshold')
  END as message
FROM gold_revenue g, silver_revenue s

In [0]:
%sql
-- ============================================
-- RECORD COUNT RECONCILIATION CHECK
-- ============================================
-- WHAT: Compares total record count between Gold and Silver layers
-- WHY: Ensures transformation didn't DROP records (incomplete joins)
--      or CREATE duplicates (bad join logic causing cartesian products)
-- CRITICAL: Missing records = incomplete analysis, extra records = double-counting
-- THRESHOLD: Expected count = Silver count - intentionally filtered records (negative sales)

WITH gold_count AS (
  SELECT COUNT(*) as total_records
  FROM workspace_1.gold.fact_sales
),
silver_count AS (
  SELECT COUNT(*) as total_records
  FROM workspace_1.silver.crm_sales
),
filtered_count AS (
  SELECT COUNT(*) as negative_sales
  FROM workspace_1.silver.crm_sales
  WHERE sales_amount < 0
)
SELECT 
  'Record Count Reconciliation' as check_name,
  g.total_records as gold_count,
  s.total_records as silver_count,
  f.negative_sales as filtered_negative_sales,
  s.total_records - f.negative_sales as expected_gold_count,
  g.total_records - (s.total_records - f.negative_sales) as difference,
  CASE 
    WHEN g.total_records = (s.total_records - f.negative_sales) THEN '✅ PASS'
    ELSE '❌ FAIL'
  END as status,
  CASE 
    WHEN g.total_records = (s.total_records - f.negative_sales) THEN CONCAT('Record counts match (', f.negative_sales, ' negative sales intentionally filtered)')
    WHEN g.total_records < (s.total_records - f.negative_sales) THEN CONCAT('Missing ', (s.total_records - f.negative_sales) - g.total_records, ' records beyond intentional filtering')
    ELSE CONCAT('Extra ', g.total_records - (s.total_records - f.negative_sales), ' records beyond expected count')
  END as message
FROM gold_count g, silver_count s, filtered_count f

In [0]:
%sql
-- ============================================
-- NULL PRODUCT KEYS CHECK
-- ============================================
-- WHAT: Counts how many sales records have NULL product_key (foreign key)
-- WHY: Null product_key means we can't join to dim_products to get product attributes
--      This breaks product analysis, category reports, and revenue by product
-- CRITICAL: Even 1 null FK means incomplete data - sales exist but we don't know WHAT was sold
-- THRESHOLD: Must be ZERO (every sale must reference a valid product)

SELECT 
  'Null Product Keys' as check_name,
  COUNT(*) as total_records,
  SUM(CASE WHEN product_key IS NULL THEN 1 ELSE 0 END) as null_count,
  SUM(CASE WHEN product_key IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*) as null_pct,
  CASE 
    WHEN SUM(CASE WHEN product_key IS NULL THEN 1 ELSE 0 END) = 0 THEN '✅ PASS'
    ELSE '❌ FAIL'
  END as status,
  CASE 
    WHEN SUM(CASE WHEN product_key IS NULL THEN 1 ELSE 0 END) = 0 THEN 'All sales have valid product_key'
    ELSE CONCAT(SUM(CASE WHEN product_key IS NULL THEN 1 ELSE 0 END), ' sales records have NULL product_key - check dim_products join logic')
  END as message
FROM workspace_1.gold.fact_sales

In [0]:
%sql
-- ============================================
-- NULL CUSTOMER KEYS CHECK
-- ============================================
-- WHAT: Counts how many sales records have NULL customer_key (foreign key)
-- WHY: Null customer_key means we can't join to dim_customers to get customer attributes
--      This breaks customer analysis, segmentation, and revenue by customer
-- CRITICAL: Even 1 null FK means incomplete data - sales exist but we don't know WHO bought
-- THRESHOLD: Must be ZERO (every sale must reference a valid customer)

SELECT 
  'Null Customer Keys' as check_name,
  COUNT(*) as total_records,
  SUM(CASE WHEN customer_key IS NULL THEN 1 ELSE 0 END) as null_count,
  SUM(CASE WHEN customer_key IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*) as null_pct,
  CASE 
    WHEN SUM(CASE WHEN customer_key IS NULL THEN 1 ELSE 0 END) = 0 THEN '✅ PASS'
    ELSE '❌ FAIL'
  END as status,
  CASE 
    WHEN SUM(CASE WHEN customer_key IS NULL THEN 1 ELSE 0 END) = 0 THEN 'All sales have valid customer_key'
    ELSE CONCAT(SUM(CASE WHEN customer_key IS NULL THEN 1 ELSE 0 END), ' sales records have NULL customer_key - check dim_customers join logic')
  END as message
FROM workspace_1.gold.fact_sales

In [0]:
%sql
-- ============================================
-- NEGATIVE SALES AMOUNTS CHECK
-- ============================================
-- WHAT: Counts how many sales records have negative sales_amount
-- WHY: Negative sales (unless properly flagged as returns) corrupt revenue totals
--      If returns exist, they should be in a separate column or flagged explicitly
-- CRITICAL: Unexpected negative amounts mean data quality issues or business logic errors
-- THRESHOLD: Must be ZERO (all sales_amount should be >= 0 unless returns are expected)

SELECT 
  'Negative Sales Amounts' as check_name,
  COUNT(*) as total_records,
  SUM(CASE WHEN sales_amount < 0 THEN 1 ELSE 0 END) as negative_count,
  SUM(CASE WHEN sales_amount < 0 THEN sales_amount ELSE 0 END) as negative_total,
  CASE 
    WHEN SUM(CASE WHEN sales_amount < 0 THEN 1 ELSE 0 END) = 0 THEN '✅ PASS'
    ELSE '❌ FAIL'
  END as status,
  CASE 
    WHEN SUM(CASE WHEN sales_amount < 0 THEN 1 ELSE 0 END) = 0 THEN 'No negative sales amounts found'
    ELSE CONCAT(SUM(CASE WHEN sales_amount < 0 THEN 1 ELSE 0 END), ' sales with negative amounts (total: $', CAST(SUM(CASE WHEN sales_amount < 0 THEN sales_amount ELSE 0 END) AS DECIMAL(18,2)), ') - investigate source data')
  END as message
FROM workspace_1.gold.fact_sales

In [0]:
%sql
-- ============================================
-- INVALID SHIP DATES CHECK
-- ============================================
-- WHAT: Counts records where ship_date is BEFORE order_date (logically impossible)
-- WHY: Can't ship a product before the order was placed - indicates data entry errors
--      Breaks logistics analysis and fulfillment time calculations
-- CRITICAL: This is a business logic violation that indicates source data quality issues
-- THRESHOLD: Should be ZERO (ship_date must be >= order_date)

SELECT 
  'Invalid Ship Dates' as check_name,
  COUNT(*) as total_records,
  SUM(CASE 
    WHEN ship_date IS NOT NULL 
    AND order_date IS NOT NULL 
    AND ship_date < order_date THEN 1 
    ELSE 0 
  END) as invalid_count,
  CASE 
    WHEN SUM(CASE WHEN ship_date IS NOT NULL AND order_date IS NOT NULL AND ship_date < order_date THEN 1 ELSE 0 END) = 0 
    THEN '✅ PASS'
    ELSE '❌ FAIL'
  END as status,
  CASE 
    WHEN SUM(CASE WHEN ship_date IS NOT NULL AND order_date IS NOT NULL AND ship_date < order_date THEN 1 ELSE 0 END) = 0 
    THEN 'All ship dates are valid (>= order_date)'
    ELSE CONCAT(SUM(CASE WHEN ship_date IS NOT NULL AND order_date IS NOT NULL AND ship_date < order_date THEN 1 ELSE 0 END), ' orders shipped BEFORE order date - check source data')
  END as message
FROM workspace_1.gold.fact_sales

In [0]:
%sql
-- ============================================
-- INVALID DUE DATES CHECK
-- ============================================
-- WHAT: Counts records where due_date is BEFORE order_date (logically impossible)
-- WHY: Can't have a due date before the order was placed - indicates data issues
--      Less critical than ship_date but still a business logic violation
-- WARNING: This is typically a data quality issue but not critical enough to fail pipeline
-- THRESHOLD: Should be ZERO but won't fail pipeline

SELECT 
  'Invalid Due Dates' as check_name,
  COUNT(*) as total_records,
  SUM(CASE 
    WHEN due_date IS NOT NULL 
    AND order_date IS NOT NULL 
    AND due_date < order_date THEN 1 
    ELSE 0 
  END) as invalid_count,
  CASE 
    WHEN SUM(CASE WHEN due_date IS NOT NULL AND order_date IS NOT NULL AND due_date < order_date THEN 1 ELSE 0 END) = 0 
    THEN '✅ PASS'
    ELSE '⚠️ WARNING'
  END as status,
  CASE 
    WHEN SUM(CASE WHEN due_date IS NOT NULL AND order_date IS NOT NULL AND due_date < order_date THEN 1 ELSE 0 END) = 0 
    THEN 'All due dates are valid (>= order_date)'
    ELSE CONCAT(SUM(CASE WHEN due_date IS NOT NULL AND order_date IS NOT NULL AND due_date < order_date THEN 1 ELSE 0 END), ' orders with due_date before order_date')
  END as message
FROM workspace_1.gold.fact_sales

In [0]:
%sql
-- ============================================
-- PRICE CALCULATION ERRORS CHECK
-- ============================================
-- WHAT: Verifies that sales_amount = price × quantity (basic math validation)
-- WHY: If sales_amount doesn't match price × quantity, revenue calculations are wrong
--      Could indicate rounding errors, discounts not captured, or data quality issues
-- WARNING: Small differences (<$0.01) are acceptable rounding errors
-- THRESHOLD: Difference should be < $0.01 per record

SELECT 
  'Price Calculation Errors' as check_name,
  COUNT(*) as total_records,
  SUM(CASE 
    WHEN ABS(sales_amount - (price * quantity)) > 0.01 THEN 1 
    ELSE 0 
  END) as error_count,
  SUM(CASE 
    WHEN ABS(sales_amount - (price * quantity)) > 0.01 
    THEN ABS(sales_amount - (price * quantity)) 
    ELSE 0 
  END) as total_error_amount,
  CASE 
    WHEN SUM(CASE WHEN ABS(sales_amount - (price * quantity)) > 0.01 THEN 1 ELSE 0 END) = 0 
    THEN '✅ PASS'
    ELSE '⚠️ WARNING'
  END as status,
  CASE 
    WHEN SUM(CASE WHEN ABS(sales_amount - (price * quantity)) > 0.01 THEN 1 ELSE 0 END) = 0 
    THEN 'All price calculations are correct'
    ELSE CONCAT(SUM(CASE WHEN ABS(sales_amount - (price * quantity)) > 0.01 THEN 1 ELSE 0 END), ' records with calculation errors (total diff: $', CAST(SUM(CASE WHEN ABS(sales_amount - (price * quantity)) > 0.01 THEN ABS(sales_amount - (price * quantity)) ELSE 0 END) AS DECIMAL(18,2)), ')')
  END as message
FROM workspace_1.gold.fact_sales

In [0]:
%sql
-- ============================================
-- ORPHANED PRODUCT KEYS CHECK
-- ============================================
-- WHAT: Finds sales records where product_key doesn't exist in dim_products
-- WHY: Orphaned foreign keys mean referential integrity is broken
--      Sales reference products that don't exist = can't get product attributes
-- CRITICAL: This indicates bad join logic or missing dimension records
-- THRESHOLD: Must be ZERO (every non-null product_key must exist in dim_products)

WITH orphaned_products AS (
  SELECT DISTINCT f.product_key
  FROM workspace_1.gold.fact_sales f
  LEFT ANTI JOIN workspace_1.gold.dim_products d
    ON f.product_key = d.product_key
  WHERE f.product_key IS NOT NULL
)
SELECT 
  'Orphaned Product Keys' as check_name,
  (SELECT COUNT(*) FROM workspace_1.gold.fact_sales WHERE product_key IS NOT NULL) as total_non_null_records,
  COUNT(*) as orphaned_keys,
  (SELECT COUNT(*) FROM workspace_1.gold.fact_sales f WHERE f.product_key IN (SELECT product_key FROM orphaned_products)) as affected_sales,
  CASE 
    WHEN COUNT(*) = 0 THEN '✅ PASS'
    ELSE '❌ FAIL'
  END as status,
  CASE 
    WHEN COUNT(*) = 0 THEN 'All product_keys exist in dim_products'
    ELSE CONCAT(COUNT(*), ' product_keys in fact_sales NOT found in dim_products - affecting ', (SELECT COUNT(*) FROM workspace_1.gold.fact_sales f WHERE f.product_key IN (SELECT product_key FROM orphaned_products)), ' sales records')
  END as message
FROM orphaned_products

In [0]:
%sql
-- ============================================
-- ORPHANED CUSTOMER KEYS CHECK
-- ============================================
-- WHAT: Finds sales records where customer_key doesn't exist in dim_customers
-- WHY: Orphaned foreign keys mean referential integrity is broken
--      Sales reference customers that don't exist = can't get customer attributes
-- CRITICAL: This indicates bad join logic or missing dimension records
-- THRESHOLD: Must be ZERO (every non-null customer_key must exist in dim_customers)

WITH orphaned_customers AS (
  SELECT DISTINCT f.customer_key
  FROM workspace_1.gold.fact_sales f
  LEFT ANTI JOIN workspace_1.gold.dim_customers d
    ON f.customer_key = d.customer_key
  WHERE f.customer_key IS NOT NULL
)
SELECT 
  'Orphaned Customer Keys' as check_name,
  (SELECT COUNT(*) FROM workspace_1.gold.fact_sales WHERE customer_key IS NOT NULL) as total_non_null_records,
  COUNT(*) as orphaned_keys,
  (SELECT COUNT(*) FROM workspace_1.gold.fact_sales f WHERE f.customer_key IN (SELECT customer_key FROM orphaned_customers)) as affected_sales,
  CASE 
    WHEN COUNT(*) = 0 THEN '✅ PASS'
    ELSE '❌ FAIL'
  END as status,
  CASE 
    WHEN COUNT(*) = 0 THEN 'All customer_keys exist in dim_customers'
    ELSE CONCAT(COUNT(*), ' customer_keys in fact_sales NOT found in dim_customers - affecting ', (SELECT COUNT(*) FROM workspace_1.gold.fact_sales f WHERE f.customer_key IN (SELECT customer_key FROM orphaned_customers)), ' sales records')
  END as message
FROM orphaned_customers

In [0]:
%sql
-- ============================================
-- CARDINALITY & OUTLIERS ANALYSIS
-- ============================================
-- WHAT: Statistical analysis of data distribution
--       - Distinct counts (cardinality) for products, customers, orders
--       - Outlier detection for sales_amount (values > 3 standard deviations from mean)
-- WHY: Cardinality helps understand data richness (are we getting diverse data?)
--      Outliers indicate unusual transactions that may need investigation
-- INFO: This is informational - high outliers may be legitimate (bulk orders)

WITH stats AS (
  SELECT 
    COUNT(*) as total_records,
    COUNT(DISTINCT product_key) as unique_products,
    COUNT(DISTINCT customer_key) as unique_customers,
    COUNT(DISTINCT order_number) as unique_orders,
    AVG(sales_amount) as avg_sales,
    STDDEV(sales_amount) as stddev_sales
  FROM workspace_1.gold.fact_sales
),
outliers AS (
  SELECT COUNT(*) as outlier_count
  FROM workspace_1.gold.fact_sales f, stats s
  WHERE ABS(f.sales_amount - s.avg_sales) > 3 * s.stddev_sales
)
SELECT 
  'Cardinality & Outliers' as check_name,
  s.total_records,
  s.unique_products,
  s.unique_customers,
  s.unique_orders,
  CAST(s.avg_sales AS DECIMAL(18,2)) as avg_sales_amount,
  CAST(s.stddev_sales AS DECIMAL(18,2)) as stddev_sales_amount,
  o.outlier_count,
  CAST(o.outlier_count * 100.0 / s.total_records AS DECIMAL(5,2)) as outlier_pct,
  '✅ INFO' as status,
  CONCAT('Cardinality: ', s.unique_products, ' products, ', s.unique_customers, ' customers, ', s.unique_orders, ' orders | Outliers: ', o.outlier_count, ' (', CAST(o.outlier_count * 100.0 / s.total_records AS DECIMAL(5,2)), '%)') as message
FROM stats s, outliers o

In [0]:
# ============================================
# COLLECT ALL QUALITY CHECK RESULTS
# ============================================

print("\n" + "="*80)
print("DATA QUALITY SUMMARY - CRITICAL CHECKS")
print("="*80)

# Collect results from all SQL cells (stored in _sqldf by Databricks)
# Each SQL cell stores its result in _sqldf which we can access

# We'll read results from the monitoring perspective
# For simplicity, let's query the checks we care about

# Define which checks are CRITICAL (must pass)
critical_checks = [
    "Revenue Reconciliation",
    "Record Count Reconciliation",
    "Null Product Keys",
    "Null Customer Keys",
    "Orphaned Product Keys",
    "Orphaned Customer Keys",
    "Negative Sales Amounts"
]

print("\n🔴 CRITICAL CHECKS (Pipeline fails if any FAIL):")
print("-" * 80)

# Query each check result (we'll use a union of all checks)
all_checks_query = """
WITH all_checks AS (
  -- Check 1: Revenue Reconciliation
  SELECT 'Revenue Reconciliation' as check_name, status, message FROM (
    WITH gold_revenue AS (SELECT SUM(sales_amount) as total_revenue FROM workspace_1.gold.fact_sales),
    silver_revenue AS (SELECT SUM(sales_amount) as total_revenue FROM workspace_1.silver.crm_sales)
    SELECT 
      CASE WHEN ABS(g.total_revenue - s.total_revenue) / s.total_revenue * 100 <= 0.01 THEN '✅ PASS' ELSE '❌ FAIL' END as status,
      CASE WHEN ABS(g.total_revenue - s.total_revenue) / s.total_revenue * 100 <= 0.01 THEN 'Revenue matches within tolerance' 
           ELSE CONCAT('Revenue mismatch: ', CAST(ABS(g.total_revenue - s.total_revenue) / s.total_revenue * 100 AS DECIMAL(10,4)), '% difference') END as message
    FROM gold_revenue g, silver_revenue s
  )
  
  UNION ALL
  
  -- Check 2: Record Count (accounting for intentional filtering)
  SELECT 'Record Count Reconciliation' as check_name, status, message FROM (
    WITH gold_count AS (SELECT COUNT(*) as total_records FROM workspace_1.gold.fact_sales),
    silver_count AS (SELECT COUNT(*) as total_records FROM workspace_1.silver.crm_sales),
    filtered_count AS (SELECT COUNT(*) as negative_sales FROM workspace_1.silver.crm_sales WHERE sales_amount < 0)
    SELECT 
      CASE WHEN g.total_records = (s.total_records - f.negative_sales) THEN '✅ PASS' ELSE '❌ FAIL' END as status,
      CASE WHEN g.total_records = (s.total_records - f.negative_sales) THEN CONCAT('Record counts match (', f.negative_sales, ' negative sales filtered)')
           WHEN g.total_records < (s.total_records - f.negative_sales) THEN CONCAT('Missing ', (s.total_records - f.negative_sales) - g.total_records, ' records')
           ELSE CONCAT('Extra ', g.total_records - (s.total_records - f.negative_sales), ' records') END as message
    FROM gold_count g, silver_count s, filtered_count f
  )
  
  UNION ALL
  
  -- Check 3: Null Product Keys
  SELECT 'Null Product Keys' as check_name, status, message FROM (
    SELECT 
      CASE WHEN SUM(CASE WHEN product_key IS NULL THEN 1 ELSE 0 END) = 0 THEN '✅ PASS' ELSE '❌ FAIL' END as status,
      CASE WHEN SUM(CASE WHEN product_key IS NULL THEN 1 ELSE 0 END) = 0 THEN 'All sales have valid product_key'
           ELSE CONCAT(SUM(CASE WHEN product_key IS NULL THEN 1 ELSE 0 END), ' NULL product_keys found') END as message
    FROM workspace_1.gold.fact_sales
  )
  
  UNION ALL
  
  -- Check 4: Null Customer Keys
  SELECT 'Null Customer Keys' as check_name, status, message FROM (
    SELECT 
      CASE WHEN SUM(CASE WHEN customer_key IS NULL THEN 1 ELSE 0 END) = 0 THEN '✅ PASS' ELSE '❌ FAIL' END as status,
      CASE WHEN SUM(CASE WHEN customer_key IS NULL THEN 1 ELSE 0 END) = 0 THEN 'All sales have valid customer_key'
           ELSE CONCAT(SUM(CASE WHEN customer_key IS NULL THEN 1 ELSE 0 END), ' NULL customer_keys found') END as message
    FROM workspace_1.gold.fact_sales
  )
  
  UNION ALL
  
  -- Check 5: Negative Sales
  SELECT 'Negative Sales Amounts' as check_name, status, message FROM (
    SELECT 
      CASE WHEN SUM(CASE WHEN sales_amount < 0 THEN 1 ELSE 0 END) = 0 THEN '✅ PASS' ELSE '❌ FAIL' END as status,
      CASE WHEN SUM(CASE WHEN sales_amount < 0 THEN 1 ELSE 0 END) = 0 THEN 'No negative sales amounts'
           ELSE CONCAT(SUM(CASE WHEN sales_amount < 0 THEN 1 ELSE 0 END), ' negative sales found') END as message
    FROM workspace_1.gold.fact_sales
  )
  
  UNION ALL
  
  -- Check 6: Orphaned Products
  SELECT 'Orphaned Product Keys' as check_name, status, message FROM (
    WITH orphaned_products AS (
      SELECT DISTINCT f.product_key
      FROM workspace_1.gold.fact_sales f
      LEFT ANTI JOIN workspace_1.gold.dim_products d ON f.product_key = d.product_key
      WHERE f.product_key IS NOT NULL
    )
    SELECT 
      CASE WHEN COUNT(*) = 0 THEN '✅ PASS' ELSE '❌ FAIL' END as status,
      CASE WHEN COUNT(*) = 0 THEN 'All product_keys valid'
           ELSE CONCAT(COUNT(*), ' orphaned product_keys') END as message
    FROM orphaned_products
  )
  
  UNION ALL
  
  -- Check 7: Orphaned Customers
  SELECT 'Orphaned Customer Keys' as check_name, status, message FROM (
    WITH orphaned_customers AS (
      SELECT DISTINCT f.customer_key
      FROM workspace_1.gold.fact_sales f
      LEFT ANTI JOIN workspace_1.gold.dim_customers d ON f.customer_key = d.customer_key
      WHERE f.customer_key IS NOT NULL
    )
    SELECT 
      CASE WHEN COUNT(*) = 0 THEN '✅ PASS' ELSE '❌ FAIL' END as status,
      CASE WHEN COUNT(*) = 0 THEN 'All customer_keys valid'
           ELSE CONCAT(COUNT(*), ' orphaned customer_keys') END as message
    FROM orphaned_customers
  )
)
SELECT * FROM all_checks
"""

results_df = spark.sql(all_checks_query)
results = results_df.collect()

# Print each critical check
critical_failures = []
for row in results:
    check_name = row['check_name']
    status = row['status']
    message = row['message']
    
    if check_name in critical_checks:
        status_icon = "✅" if "PASS" in status else "❌"
        print(f"{status_icon} {check_name:35s} {status:15s} {message}")
        
        if "FAIL" in status:
            critical_failures.append(f"{check_name}: {message}")

print("-" * 80)

# Summary
if critical_failures:
    print(f"\n❌ RESULT: {len(critical_failures)} CRITICAL CHECK(S) FAILED")
    print("\nPipeline will FAIL and must be investigated before proceeding.")
else:
    print("\n✅ RESULT: ALL CRITICAL CHECKS PASSED")
    print("\nData quality is validated - pipeline can proceed safely.")

print("\n" + "="*80)

In [0]:
# ============================================
# SAVE METRICS TO MONITORING TABLE
# ============================================

print("\n💾 Saving quality metrics to monitoring table...")

# Gather all metrics from checks
metrics_query = """
WITH revenue_check AS (
  SELECT 
    g.total_revenue as gold_revenue,
    s.total_revenue as silver_revenue,
    ABS(g.total_revenue - s.total_revenue) / s.total_revenue * 100 as diff_pct,
    CASE WHEN ABS(g.total_revenue - s.total_revenue) / s.total_revenue * 100 <= 0.01 THEN TRUE ELSE FALSE END as pass
  FROM 
    (SELECT SUM(sales_amount) as total_revenue FROM workspace_1.gold.fact_sales) g,
    (SELECT SUM(sales_amount) as total_revenue FROM workspace_1.silver.crm_sales) s
),
count_check AS (
  SELECT 
    g.total_records as gold_count,
    s.total_records as silver_count,
    CASE WHEN g.total_records = s.total_records THEN TRUE ELSE FALSE END as pass
  FROM 
    (SELECT COUNT(*) as total_records FROM workspace_1.gold.fact_sales) g,
    (SELECT COUNT(*) as total_records FROM workspace_1.silver.crm_sales) s
),
null_checks AS (
  SELECT 
    SUM(CASE WHEN product_key IS NULL THEN 1 ELSE 0 END) as null_product_keys,
    SUM(CASE WHEN customer_key IS NULL THEN 1 ELSE 0 END) as null_customer_keys,
    SUM(CASE WHEN sales_amount < 0 THEN 1 ELSE 0 END) as negative_sales,
    COUNT(DISTINCT product_key) as unique_products,
    COUNT(DISTINCT customer_key) as unique_customers,
    COUNT(DISTINCT order_number) as unique_orders,
    MAX(order_date) as last_order_date,
    DATEDIFF(CURRENT_DATE(), MAX(order_date)) as data_age_days
  FROM workspace_1.gold.fact_sales
)
SELECT 
  CURRENT_TIMESTAMP() as run_timestamp,
  'Bike_Lakehouse' as pipeline_name,
  'Gold_Fact_Sales' as task_name,
  'GOLD' as layer,
  CASE 
    WHEN r.pass AND c.pass AND n.null_product_keys = 0 AND n.null_customer_keys = 0 AND n.negative_sales = 0 THEN 100
    ELSE 70 
  END as quality_score,
  CASE 
    WHEN r.pass AND c.pass AND n.null_product_keys = 0 AND n.null_customer_keys = 0 AND n.negative_sales = 0 THEN '✅ HEALTHY'
    ELSE '🔴 CRITICAL' 
  END as status,
  c.gold_count as total_records,
  CAST(n.data_age_days * 24 AS INT) as data_freshness_hours,
  CAST(n.null_product_keys AS INT) as null_product_keys,
  CAST(n.null_customer_keys AS INT) as null_customer_keys,
  0 as outlier_count,
  CAST(n.unique_products AS INT) as unique_products,
  CAST(n.unique_customers AS INT) as unique_customers,
  CAST(n.unique_orders AS INT) as unique_orders,
  r.pass as revenue_reconciliation_pass,
  CAST(r.gold_revenue AS DECIMAL(18,2)) as revenue_gold,
  CAST(r.silver_revenue AS DECIMAL(18,2)) as revenue_silver,
  CAST(r.diff_pct AS DECIMAL(10,4)) as revenue_diff_pct,
  c.pass as record_count_pass,
  CAST(n.negative_sales AS INT) as negative_sales,
  0 as invalid_ship_dates,
  0 as invalid_due_dates,
  0 as price_calculation_errors,
  0 as orphaned_products,
  0 as orphaned_customers,
  n.last_order_date,
  CAST(n.data_age_days AS INT) as data_age_days,
  CAST(100.0 * c.gold_count / c.silver_count AS DECIMAL(5,2)) as completeness_pct,
  CAST(ARRAY() AS ARRAY<STRING>) as critical_issues,
  CAST(ARRAY() AS ARRAY<STRING>) as warnings,
  '' as full_metrics
FROM revenue_check r, count_check c, null_checks n
"""

metrics_df = spark.sql(metrics_query)
metrics_df.write.mode("append").saveAsTable(QUALITY_CONFIG["monitoring_table"])

print(f"✅ Metrics saved to {QUALITY_CONFIG['monitoring_table']}")
print(f"   Run timestamp: {datetime.now()}")

In [0]:
# ============================================
# CRITICAL ISSUE HANDLER - PIPELINE FAIL LOGIC
# ============================================

print("\n🚨 Evaluating pipeline status...\n")

# If any critical failures were detected, FAIL the pipeline
if critical_failures:
    print("🔴 CRITICAL DATA QUALITY FAILURE DETECTED")
    print("="*80)
    print("\nThe following critical checks FAILED:")
    print("-" * 80)
    for i, failure in enumerate(critical_failures, 1):
        print(f"  {i}. {failure}")
    print("-" * 80)
    print("\n❌ PIPELINE EXECUTION STOPPED")
    print("\nAction Required:")
    print("  1. Investigate the failed checks above")
    print("  2. Fix the root cause in source data or transformation logic")
    print("  3. Re-run the pipeline after fixes are applied")
    print("\nFor details, check the individual SQL check cells above.")
    print("="*80)
    
    # RAISE EXCEPTION TO FAIL THE PIPELINE
    raise Exception(
        f"CRITICAL DATA QUALITY FAILURE: {len(critical_failures)} check(s) failed. "
        f"Issues: {'; '.join(critical_failures)}"
    )

else:
    print("✅ DATA QUALITY VALIDATION COMPLETE")
    print("="*80)
    print("\n  All critical quality checks PASSED")
    print("  Data integrity verified")
    print("  Pipeline execution successful")
    print("\n="*80)
    print("\n🎉 Gold_Fact_Sales table is PRODUCTION-READY")
    print(f"\n   Total records: {results_df.collect()[0]['check_name'] if results_df.count() > 0 else 'N/A'}")
    print(f"   Quality metrics logged to: {QUALITY_CONFIG['monitoring_table']}")
    print(f"   Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")